# Day-to-day evolution of supply and demand
Module for simulating ridesourcing evolution, including pooled rides

Contribution by Arjan de Ruijter - a.j.f.deruijter@tudelft.nl

In [4]:
%load_ext autoreload
%autoreload 2
import os, sys # add MaaSSim and MaaSSim/MaaSSim to path (not needed if already in path)
module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
from MaaSSim.utils import save_config, get_config, load_G, generate_demand, initialize_df, empty_series, \
    slice_space, test_space, read_requests_csv
from MaaSSim.maassim import Simulator
from MaaSSim.data_structures import structures as inData
from MaaSSim.d2d_sim import *
from MaaSSim.d2d_demand import *
from MaaSSim.d2d_supply import *
from MaaSSim.d2d_shared import prep_shared_rides
from MaaSSim.decisions import dummy_False

In [6]:
import pandas as pd
import zipfile
import logging
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.lines import Line2D
import numpy as np
import random
import ExMAS
plt.style.use('ggplot')
np.random.seed(0)
random.seed(0)

In [7]:
# Load config
params = get_config('../../data/config/delft.json')  # load configuration
params.paths.albatross = '../../data/albatross'

In [8]:
# Experiment replications and number of threads to be used
params.parallel.nReplications = 1
params.parallel.nThread = 1

# Main experimental settings
params.nP = 10000 # travellers
params.nV = 20 # drivers
params.nD = 1 # days
params.simTime = 8 # hours

In [9]:
# Other day-to-day settings
params.evol.drivers.kappa = 0.2 # learning weight (supply-side)
params.evol.drivers.res_wage.mean = 25 #euros/h
params.evol.drivers.gini = 0.35 # gini coefficient used to establish sigma parameter of log-norm distribution of res wage
params.evol.drivers.init_inc_ratio = 1 #expected income of informed drivers at start of sim as ratio of res wage

params.evol.drivers.inform.prob_start = 1 # probability of being informed at start of sim
params.evol.drivers.inform.beta = 0.1 # information transmission rate
params.evol.drivers.inform.std_fact = 0.5 # multiplier of the standard deviation of experienced income used in signal

params.evol.drivers.regist.prob_start = 1 # probability of being registered if informed at start of sim
params.evol.drivers.regist.beta = 0.2 # registration choice model parameter
params.evol.drivers.regist.cost_comp = 20 # daily share of registration costs (euros)
params.evol.drivers.regist.samp = 0.5 # probability of making (de)regist decision
params.evol.drivers.regist.min_work_exp = 0 # Working experience required before deregistration is possible
params.evol.drivers.regist.min_days = 5 # Minimum number of registered days before driver can deregister

params.evol.drivers.particip.beta = 0.1 # participation choice model parameter
params.evol.drivers.particip.probabilistic = True # stochasticity in participation choice

params.evol.travellers.inform.prob_start = 1 # probability that traveller is informed at start of sim
params.evol.travellers.inform.beta = 0.1 # information transmission rate (demand-side)
params.evol.travellers.inform.start_wait = 0 # expected waiting time at start of simulation
params.evol.travellers.inform.std_fact = 0.5 # multiplier of the standard deviation of experienced waiting time used in signal
params.evol.travellers.reject_penalty = 30 * 60 # seconds
params.evol.travellers.kappa = 0.2 # learning weight (demand-side)
params.evol.travellers.min_prob = 0.05 # filtering criterion, when probability is lower when waiting time is zero, never consider RS

params.evol.travellers.mode_pref.mean_vot = 10 # Mean VoT in euro/h
params.evol.travellers.mode_pref.access_multip = 2 # Multiplier of access time compared to in-vehicle time
params.evol.travellers.mode_pref.wait_multip = 2.5 # Multiplier of waiting time compared to in-vehicle time
params.evol.travellers.mode_pref.bike_multip = 2 # Multiplier of biking time compared to in-vehicle time
params.evol.travellers.mode_pref.beta_cost = -0.1592 # util/euro
params.evol.travellers.mode_pref.transfer_pen = 5 * 60 # seconds, to be added to IVT for each transfer
params.evol.travellers.mode_pref.ASC_car = 0 # util, rel to bike
params.evol.travellers.mode_pref.ASC_rs = 0
params.evol.travellers.mode_pref.ASC_pt =  0
params.evol.travellers.mode_pref.ASC_car_sd = 0 # standard deviation in ASCs
params.evol.travellers.mode_pref.ASC_rs_sd = 0
params.evol.travellers.mode_pref.ASC_pt_sd = 0
params.evol.travellers.mode_pref.ASC_bike_sd = 0
params.evol.travellers.mode_pref.gini = params.evol.drivers.gini

# Financial settings
params.platforms.base_fare = 1 #euro
params.platforms.fare = 0 #euro/km
params.platforms.min_fare = 0 # euro
params.platforms.comm_rate = 0.05 #rate
params.drivers.fuel_costs = 0.25 #euro/km

# Properties alternative modes
params.alt_modes.pt.option = False
params.alt_modes.pt.base_fare = 0.99 # euro
params.alt_modes.pt.km_fare = 0.174 # euro/km
params.alt_modes.car.km_cost = 0.5 # euro/km
params.alt_modes.car.diff_parking = True # different parking tariffs in city
params.alt_modes.car.park_cost = 7.5 # euro
params.alt_modes.car.park_cost_center = 15 # euro
params.alt_modes.car.access_time = 10 * 60 # s
params.speeds.bike = (1/2.5) * params.speeds.ride # m/s

# Regulation
params.platforms.reg_cap = np.inf # registration cap
params.platforms.ptcp_cap = np.inf # daily participation cap

# Demand settings
# params.demand_structure.origins_dispertion = -0.0003
# params.demand_structure.destinations_dispertion = -0.0003
params.dist_threshold_min = 2000 # min dist
# params.dist_threshold = 100000 # max dist

# Start time
# params.t0 = pd.Timestamp.now()
params.t0 = pd.Timestamp(2021, 11, 1, 9)

In [10]:
# Pooling settings
params.shareability.offered = True
params.shareability.avg_speed = params.speeds.ride
params.shareability.min_discount = 0.5
params.shareability.add_discount = 0.0
params.shareability.shared_discount = params.shareability.min_discount
params.shareability.delay_value = 1
params.shareability.WtS = 1.1 # Willingness to share
params.shareability.price = 1.5 #eur/km
params.shareability.VoT = 0.0035 #eur/s
params.shareability.matching_obj = 'u_pax' #minimize VHT for vehicles
params.shareability.pax_delay = 0
params.shareability.horizon = 600
params.shareability.max_degree = 2
params.shareability.nP = params.nP
params.shareability.share = 1
params.shareability.without_matching = True

In [11]:
inData = load_G(inData, params, stats=True, set_t=False)  # download graph for the 'params.city' and calc the skim matrices
if params.alt_modes.car.diff_parking:
    inData = diff_parking(inData) # determine which nodes are in center

In [12]:
inData = generate_demand(inData, params, avg_speed = True)

In [10]:
# Load processed Albatross file, the OTP result, and compute PT fares
# inData = load_albatross_proc(inData, params, avg_speed = True)
# inData.requests = inData.requests.drop(['orig_geo', 'dest_geo', 'origin_y', 'origin_x', 'destination_y', 'destination_x', 'time'], axis = 1)
# inData.pt_itinerary = load_OTP_result(params)
# inData = consist_OTP_alba(inData, params)

In [11]:
# Prepare supply and demand attributes
inData.passengers = prefs_travs(inData, params)
all_pax = mode_filter(inData, params)
inData.passengers = all_pax[all_pax.mode_choice == "day-to-day"]
inData.requests = inData.requests[inData.requests.pax_id.isin(inData.passengers.index)]
if params.alt_modes.pt.option:
    inData.pt_itinerary = inData.pt_itinerary[inData.pt_itinerary.pax_id.isin(inData.passengers.index)]
    inData.passengers.reset_index(drop=True, inplace=True)
    inData.requests.reset_index(drop=True, inplace=True)
    inData.pt_itinerary.reset_index(drop=True, inplace=True)
inData.requests['pax_id'] = inData.requests.index
inData.pt_itinerary['pax_id'] = inData.pt_itinerary.index
inData.passengers['informed'] = np.random.rand(len(inData.passengers)) < params.evol.travellers.inform.prob_start
inData.passengers['expected_wait'] = params.evol.travellers.inform.start_wait
inData.passengers['expected_wait_pool'] = params.evol.travellers.inform.start_wait
inData.passengers['expected_pool_discount'] = params.shareability.min_discount
inData.passengers['expected_pool_delay'] = 0
fixed_supply = generate_vehicles_d2d(inData, params)
inData.vehicles = fixed_supply.copy()
inData.vehicles.platform = inData.vehicles.apply(lambda x: 0, axis = 1)
inData.passengers.platforms = inData.passengers.apply(lambda x: [0], axis = 1)
inData.requests['platform'] = inData.requests.apply(lambda row: inData.passengers.loc[row.name].platforms[0], axis = 1) 
inData.platforms = pd.concat([inData.platforms,pd.DataFrame(columns=['base_fare','comm_rate','min_fare'])])
inData.platforms = initialize_df(inData.platforms)
inData.platforms.loc[0]=[params.platforms.fare,'Uber',30,params.platforms.base_fare,params.platforms.comm_rate,params.platforms.min_fare,]

In [ ]:
inData = ExMAS.main(inData, params.shareability, plot=False) # create shareability graph (ExMAS) 

03-03-23 11:46:40-INFO-Initializing pairwise trip shareability between 10000 and 10000 trips.
03-03-23 11:46:41-INFO-creating combinations


In [ ]:
# Day-to-day simulation (incl. processing)
sim = Simulator(inData, params=params,
                    kpi_veh = D2D_veh_exp,
                    kpi_pax = d2d_kpi_pax,
                    f_driver_out = D2D_driver_out,
                    f_trav_out = d2d_no_request,
                    f_trav_mode = dummy_False,
                    logger_level=logging.WARNING)  # initialize

evol_micro = init_d2d_dotmap()
for day in range(params.get('nD', 1)):  # run iterations
    inData.passengers = mode_preday(inData, params)
    temp_rides = inData.sblts.rides.copy()
    temp_reqs = inData.sblts.requests.copy()
    
    rs_users = inData.passengers[(inData.passengers.mode_day == 'rs') | (inData.passengers.mode_day == 'pool')].index.tolist()
    poolers = inData.passengers[inData.passengers.mode_day == 'pool'].index.tolist()
    inData.sblts.rides = inData.sblts.rides[inData.sblts.rides.apply(lambda x: all(i in rs_users for i in x.indexes), axis=1)] # filter out all travellers opting for mode outside ride-hailing market
    inData.sblts.rides = inData.sblts.rides[inData.sblts.rides.apply(lambda x: (all(i in poolers for i in x.indexes) or x.kind == 1), axis=1)]  # filter out pooled trips for individuals opting for private ride
    inData.sblts.requests = inData.sblts.requests[inData.sblts.requests.apply(lambda x: x.pax_id in rs_users, axis=1)]
    
    inData = prep_shared_rides(inData, params.shareability)  # prepare schedules
    sim.make_and_run(run_id=day)  # prepare and SIM
    sim.output()  # calc results
    sim.last_res = sim.res[day].copy()
    del sim.res[day]

    drivers_summary = update_d2d_drivers(sim=sim,params=params)
    travs_summary = update_d2d_travellers(sim=sim,params=params)
    
    exp_df = update_work_exp(inData, drivers_summary)
    inData.vehicles.work_exp = exp_df.work_exp
    inData.days_since_reg = exp_df.days_since_reg

    res_inf_driver = wom_driver(inData, params = params)
    inData.vehicles.informed = res_inf_driver
    inData.vehicles.expected_income = learning_unregist(inData, drivers_summary, params = params)
    
    res_regist = platform_regist(inData, drivers_summary, params = params)
    inData.vehicles.registered = res_regist.registered
    inData.vehicles.work_exp = res_regist.work_exp
    inData.vehicles.pos = fixed_supply.pos
    inData.vehicles.rejected_reg = res_regist.rejected_reg
    exp_inf_trav = travs_summary.loc[travs_summary.informed]
    average_xp_wait = exp_inf_trav.corr_xp_wait.mean() / 60
    res_inf_trav = wom_trav(inData, travs_summary, params = params)
    inData.passengers.informed = res_inf_trav.informed
    inData.passengers.expected_wait = res_inf_trav.perc_wait
    
    inData.sblts.rides = temp_rides.copy()
    inData.sblts.requests = temp_reqs.copy()
    
    evol_micro = d2d_summary_day(evol_micro, drivers_summary, travs_summary, day)

In [ ]:
evol_micro, evol_agg = d2d_agg_statistics(evol_micro, params) # multi-day stats

In [ ]:
# Save d2d stats to zip file
with zipfile.ZipFile('evol.zip', 'w') as csv_zip:
    csv_zip.writestr("evol_agg_supply.csv", evol_agg.supply.to_csv())
    csv_zip.writestr("evol_agg_demand.csv", evol_agg.demand.to_csv())

In [ ]:
evol_agg.supply

In [ ]:
evol_agg.demand

In [ ]:
# Plot number of drivers and income
fig, axes = plt.subplots(nrows=5, ncols=1, figsize = (8,12.5), sharex = True)
evol_agg.supply[['inform','regist','particip']].plot(ax = axes[0], color=['lightsteelblue','tab:blue','midnightblue'])
axes[0].set_title('(A) Ridesourcing supply')
axes[0].legend(['Informed','Registered','Participating'])
axes[0].set_ylim([0,params.nV + 25])
axes[0].set_ylabel('Number of drivers')
evol_agg.supply[['mean_perc_inc','mean_exp_inc']].plot(ax = axes[2], color=['lightsteelblue','midnightblue'])
axes[2].set_title('(C) Driver earnings')
axes[2].legend(['Expected','Experienced'])
axes[2].set_ylim([0,math.ceil(max(evol_agg.supply.mean_perc_inc.max(),evol_agg.supply.mean_exp_inc.max())/50)*50])
axes[2].set_ylabel('Income (\u20ac)')


evol_agg.demand[['requests','bike','car','pt']].plot.area(ax = axes[1])
h,l = axes[1].get_legend_handles_labels()
evol_agg.demand['inform'].plot(ax = axes[1], color = 'black', linestyle = 'dashed', label = 'informed')
line = Line2D([0], [0],color='black', linestyle ='dashed')
axes[1].set_title('(B) Demand')
axes[1].set_ylim([0,len(inData.passengers) * 1.1])
axes[1].set_ylabel('Number of travellers')
h.extend([line])
axes[1].legend(labels=["Ridesourcing","Bike","Car","Public transport","Informed"], handles=h)

ax_sec = axes[3].twinx()
evol_agg.demand['proport_match'] = evol_agg.demand.gets_offer / evol_agg.demand.requests * 100
evol_agg.demand['mean_wait'].apply(lambda x: 1/60 * x).plot(ax = axes[3], label='Experienced waiting time', color ='lightsalmon')
evol_agg.demand['corr_mean_wait'].apply(lambda x: 1/60 * x).plot(ax = axes[3], label='Corrected exp. waiting time', color ='rosybrown')
evol_agg.demand['perc_wait'].apply(lambda x: 1/60 * x).plot(ax = axes[3], label='Expected waiting time', color ='tomato')
evol_agg.demand['proport_match'].plot(ax = ax_sec, color = 'grey', label='Share of requests returned with offer', linestyle = 'dotted')
lines_1, labels_1 = axes[3].get_legend_handles_labels()
lines_2, labels_2 = ax_sec.get_legend_handles_labels()
lines = lines_1 + lines_2
labels = labels_1 + labels_2
axes[3].legend(lines, labels, loc=0)
axes[3].set_title('(D) Level of service')
axes[3].set_ylabel('Mean wait. time (min)')
ax_sec.set_ylabel('Request success')
ax_sec.yaxis.set_major_formatter(mtick.PercentFormatter())
max_val = max(evol_agg.demand.mean_wait.max(),evol_agg.demand.corr_mean_wait.max(),evol_agg.demand.perc_wait.max())
axes[3].set_ylim([0,math.ceil((1/60)*max_val/2)*2+0.5])
ax_sec.set_ylim([0,100+5])
ax_sec.grid(None)

proport_rs = evol_agg.demand.requests / evol_agg.demand.inform * 100 
proport_rs.plot(ax = axes[4], label = 'Travellers - Mode share of ridesourcing', color='maroon')
axes[4].set_title('(E) Platform utilisation')
axes[4].set_ylim([0,100+2])
axes[4].set_ylabel('Platform utilisation')

axes[4].yaxis.set_major_formatter(mtick.PercentFormatter())

proport_work = evol_agg.supply.particip / evol_agg.supply.regist * 100
proport_work.plot(ax = axes[4], label = 'Drivers - Labour participation rate', color ='midnightblue')
lines, labels = axes[4].get_legend_handles_labels()
axes[4].legend(labels)

plt.savefig('d2d-evo.png')

---

In [ ]:
inData.keys()

In [ ]:
# inData.pt_itinerary.to_csv('inData_pt-itinerary.csv')
inData.vehicles

In [ ]:
inData.stats.center

In [ ]:
evol_micro.supply.inform

In [ ]:
inData.skim.iloc[:20, :20].to_csv('inData_skim.csv')

In [ ]:
(evol_micro.demand.gets_offer * evol_micro.demand.requests).sum()

In [ ]:
inData.passengers.to_csv('inData_passengers.csv')

In [ ]:
evol_micro.demand.gets_offer

In [ ]:
inData.vehicles

In [ ]:
# sim.last_res.pax_exp.head(50).to_csv('pax_exp.csv')
sim.last_res.keys()

In [ ]:
# sim.last_res.veh_kpi.head(50).to_csv('veh_kpi.csv')
sim.last_res.veh_exp

In [ ]:
sim.last_res.pax_exp

In [ ]:
import networkx as nx
df = nx.to_pandas_edgelist(inData.G)
df.to_csv('edges.csv')

In [ ]:
inData.pt_itinerary

In [ ]:
abc

In [ ]:
defg

In [ ]:
inData.keys()

In [ ]:
inData.sblts.keys()

In [ ]:
inData.sblts.requests

In [ ]:
inData.requests

In [ ]:
inData.the_skim.sort_index().sort_index(axis=1).head(20)

In [ ]:
inData.sblts.requests

In [ ]:
inData.sblts.R[2]

In [ ]:
inData.sblts.rides

In [ ]:
inData.sblts.requests

In [ ]:
sim.runs[0]['trips'][sim.runs[0]['trips']['pax'] == 1722]

In [ ]:
inData.requests[inData.requests['pax_id'] == 1722]['sim_schedule']

In [ ]:
inData.requests.loc[439].sim_schedule

In [ ]:
# inData.requests.loc[421]
inData.requests.loc[13].sim_schedule

In [ ]:
inData.keys()

In [ ]:
inData.sblts.keys()

In [ ]:
inData.sblts.rides[inData.sblts.rides.selected == 1].tail(110)

In [ ]:
inData.sblts.requests.loc[21]

In [ ]:
sim.runs[1].rides

In [ ]:
sim.runs[0].rides[sim.runs[0].rides['event']=='DEPARTS_FROM_PICKUP'].head(50)

In [ ]:
# sim.runs[0].rides[sim.runs[0].rides['t']>=4000].head(50)
sim.runs[0].rides.head(50)

In [ ]:
sim.runs[0].trips[sim.runs[0].trips['pax']==19]

In [ ]:
inData.sblts.schedule['sim_schedule']

In [ ]:
inData.requests['sim_schedule'].head(50)

In [ ]:
inData.requests.sim_schedule.loc[20]

In [ ]:
inData.keys()

In [ ]:
inData.vehicles

In [ ]:
inData.passengers

In [ ]:
sim.runs[0].rides.head(50)

In [ ]:
sim.runs[0].trips[sim.runs[0].trips['pax']==111]

In [ ]:
sim.runs[0].trips[sim.runs[0].trips['pax']==104]

In [ ]:
inData.sblts.keys()

In [ ]:
inData.requests.loc[104]

In [ ]:
inData.requests.loc[104].sim_schedule

In [ ]:
inData.requests

In [ ]:
inData.requests.loc[407].sim_schedule

In [ ]:
inData.sblts.SINGLES

In [ ]:
inData.passengers.loc[201]

In [ ]:
inData.sblts.requests.loc[201]

In [ ]:
inData.sblts.rides

In [ ]:
inData.passengers[inData.passengers.mode_day == 'pool'].index.tolist()

In [ ]:
# inData.sblts.rides[inData.sblts.rides.apply(lambda x: any(i in pool for i in x.indexes))]
pool=[2,5]
inData.sblts.rides.apply(lambda x: any(i in pool for i in x.indexes), axis=1)

In [ ]:
inData.passengers.loc[422]

In [ ]:
inData.sblts.requests

In [ ]:
inData.sblts.schedule.head(50)

In [ ]:
travs_summary

In [ ]:
evol_micro.demand.wait_time

In [ ]:
evol_micro.demand.corr_wait_time

In [ ]:
evol_micro.demand.req_pool

In [ ]:
evol_micro.demand.perc_wait

In [ ]:
sim.runs[2].trips.head(50)

In [ ]:
inData.sblts.rides.head(90)

In [ ]:
inData.sblts.requests.loc[453]

In [ ]:
xyz = inData.sblts.rides[inData.sblts.rides.apply(lambda x: all(i in rs_users for i in x.indexes), axis=1)] # filter out all travellers opting for mode outside ride-hailing market
xyz = xyz[xyz.apply(lambda x: (all(i in poolers for i in x.indexes) or x.kind == 1), axis=1)]  # filter out pooled trips for individuals opting for private ride

xyz
# poolers

In [ ]:
inData.sblts.schedule.tail(50)

In [ ]:
inData.requests.loc[489]

In [ ]:
inData.requests.loc[256].sim_schedule

In [ ]:
inData.passengers.loc[256]

In [ ]:
inData.sblts.rides

In [ ]:
abc

In [ ]:
inData.sblts.schedule.loc[9084].sim_schedule

In [ ]:
inData.sblts.schedule

In [ ]:
inData.passengers.loc[256]

In [ ]:
hij

In [ ]:
klm

In [ ]:
sim.runs[0].trips[sim.runs[0].trips.pax == 383]

In [ ]:
sim.runs[0].trips[sim.runs[0].trips.pax == 484]

In [ ]:
sim.runs[2].rides[sim.runs[2].rides.veh == 6].tail(100)

In [ ]:
sim.runs[2].rides[sim.runs[2].rides.apply(lambda x: x.paxes == [119], axis=1)]

In [ ]:
inData.sblts.schedule.loc[9057].sim_schedule

In [ ]:
klm.loc[119]

In [ ]:
hij.loc[9084].

In [ ]:
inData.sblts.schedule.loc[9084]

In [ ]:
koekoek = inData.requests.copy()
koekoek['hoi'] = inData.sblts.requests.shareable
koekoek

In [ ]:
koekoek.hoi.fillna(False)

In [ ]:
travs_summary.loc[463]

In [ ]:
sim.last_res.pax_exp.loc[463]

In [ ]:
(inData.requests.loc[378].treq - params.t0).seconds

In [ ]:
evol_micro.supply

In [ ]:
sim.runs[0].outcomes

In [ ]:
abc = travs_summary[travs_summary.chosen_mode == 'pool']
abc[abc.gets_offer]

In [ ]:
travs_summary[travs_summary.chosen_mode == 'pool'].tail(50)

In [ ]:
sim.last_res.pax_exp.tail(50)

In [ ]:
inData.requests.loc[1].sim_schedule

In [ ]:
abc = travs_summary.copy()
abc['act_shared'] = abc.apply(lambda x: True if (len(inData.requests.loc[x.name].sim_schedule.req_id.dropna().unique()) > 1) else False, axis=1)
abc.loc[290]

In [ ]:
len(inData.requests.loc[290].sim_schedule.req_id.dropna().unique())

In [ ]:
inData.sblts.schedule

In [ ]:
inData.requests.loc[359].sim_schedule

In [ ]:
inData.req

In [ ]:
travs_summary[travs_summary.chosen_mode == 'pool'].tail(50)

In [ ]:
inData.requests.loc[409].sim_schedule

In [ ]:
travs_summary.chosen_mode == 'pool'

In [ ]:
xyz = inData.sblts.schedule.copy()
xyz['pooling_reqs'] = xyz.apply(lambda x: any(i in travs_summary[travs_summary.chosen_mode == 'pool'].index.to_list() for i in x.indexes) and travs_summary.gets_offer.loc[x.indexes[0]], axis=1)
xyz[xyz.pooling_reqs]
# xyz['something_weird'] = xyz.apply(lambda x: int(travs_summary.gets_offer.loc[x.indexes[0]]) + int(travs_summary.gets_offer.loc[x.indexes[-1]]), axis=1)
# xyz[xyz.something_weird == 1]

In [14]:
inData.requests.to_csv('Delft_requests.csv')

In [15]:
inData.passengers.to_csv('Delft_passengers.csv')

In [16]:
inData.platforms

,fare,name,batch_time
id,,,
